In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Read the binary file as uint8 (1 byte per bit)
data = np.fromfile("/home/bri/workspace/mode_s_decode/controlpathsdotcom/raw_adsb_files/adsb_raw_4000000Hz_1783773546.real_float", dtype=np.uint8)

# Plot as a step plot (like a digital signal)
plt.figure(figsize=(14, 4))
plt.step(range(len(data)), data, where='post', color='blue', linewidth=1.5)
plt.title("Binary Bitstream (0s and 1s) - From GNU Radio Threshold")
plt.xlabel("Bit Index")
plt.ylabel("Bit Value (0 or 1)")
plt.yticks([0, 1])
plt.grid(True, axis='y', alpha=0.3)
plt.xlim(0, len(data))  # Show full length
plt.tight_layout()
plt.show()

In [ ]:
# Show first 100 bits
start = 0
end = 50000
span = end - start
plt.figure(figsize=(10, 3))
plt.step(range(span), data[start:end], where='post', color='red')
plt.title("First 100 Bits")
plt.xlabel("Bit Index")
plt.ylabel("Bit Value")
plt.yticks([0, 1])
plt.grid(True)
plt.show()

In [ ]:
print(f"Total bits: {len(data)}")
print(f"Number of 1s: {np.sum(data)}")
print(f"Number of 0s: {len(data) - np.sum(data)}")
print(f"Percentage of 1s: {np.mean(data) * 100:.1f}%")

In [ ]:
"""
Embedded Python Blocks:

Each time this file is saved, GRC will instantiate the first class it finds
to get ports and parameters of your block. The arguments to __init__  will
be the parameters. All of them are required to have default values!
"""

import numpy as np

preamble_found = False
file = '/home/bri/workspace/mode_s_decode/controlpathsdotcom/raw_adsb_files/adsb_raw_4000000Hz_1783773546.real_float'
preamble = "100000101000000"
# def __init__(self, file=''):  # only default arguments here
#     """arguments to this function show up as parameters in GRC"""
#     gr.sync_block.__init__(
#         self,
#         name='Binary File Processor',   # will show up in GRC
#         in_sig=[np.float32],
#         out_sig=[np.float32]
#     )
#     # if an attribute with the same name as a parameter is found,
#     # a callback is registered (properties work, too).
#     self.filename = file
#1010001010000000
def find_sequence_in_binary_file(filename, sequence):
    """Check a binary file for the presence of a specific bit sequence.
    
    Args:
        filename (str): Path to the binary file to check
        sequence (str): The bit sequence to search for (e.g., "1010001010000000")
    
    Returns:
        list: List of byte offsets where the sequence starts, or empty list if not found
    """
    if not sequence or not all(c in '01' for c in sequence):
        raise ValueError("Sequence must be a string of 0s and 1s (e.g., '10100010')")

    # Convert the bit sequence to bytes (pad to full bytes)
    # e.g., "1010001010000000" → 2 bytes
    num_bytes = (len(sequence) + 7) // 8
    # Pad with leading zeros if needed
    padded_sequence = sequence.zfill(num_bytes * 8)
    sequence_int = int(padded_sequence, 2)
    sequence_bytes = sequence_int.to_bytes(num_bytes, byteorder='big')

    # Also keep the original bit string for verification
    sequence_bits = sequence

    found_positions = []

    try:
        with open(filename, 'rb') as file:
            chunk_size = 65536  # 64 KB chunks
            buffer = b''
            buffer_bits = ""  # Keep track of bits for overlap

            while True:
                chunk = file.read(chunk_size)
                if not chunk:
                    break

                # Convert chunk to binary string (e.g., b'\x01\x02' → "0000000100000010")
                chunk_bits = ''.join(format(byte, '08b') for byte in chunk)

                # Combine with previous buffer's bits (for overlap)
                combined_bits = buffer_bits + chunk_bits

                # Search for the sequence in the combined bits
                start = 0
                while True:
                    pos = combined_bits.find(sequence_bits, start)
                    if pos == -1:
                        break
                    # Convert bit position to byte position
                    byte_pos = (len(buffer) * 8 + pos) // 8
                    found_positions.append(byte_pos)
                    start = pos + 1  # Continue searching for overlapping matches
                    preamble_found = True

                # Update buffer for next iteration (keep enough for overlap)
                buffer_bits = combined_bits[-(len(sequence_bits) - 1):]
                buffer = chunk  # Keep full chunk for byte position calculation

        return found_positions

    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []

# def find_sequence_in_binary_file(filename, sequence="1010001010000000"):
#     """ Check a binary file for the presence of a specific bit sequence.
    
#     Args:
#         filename (str): Path to the binary file to check
#         sequence (str): The bit sequence to search for (default: "10011010")
    
#     Returns:
#         bool: True if sequence is found, False otherwise
#     """
#     global preamble_found

#     # Convert sequence to bytes for comparison
#     sequence_bytes = int(sequence, 2).to_bytes(1, byteorder='big')
    
#     try:
#         with open(filename, 'rb') as file:
#             # Read the file in chunks to handle large files efficiently
#             chunk_size = 1024
#             buffer = b''
            
#             while True:
#                 chunk = file.read(chunk_size)
#                 if not chunk:
#                     break
                
#                 # Append new data to buffer
#                 buffer += chunk
                
#                 # Check for the sequence in the buffer
#                 # We need to check overlapping sequences
#                 for i in range(len(buffer) - len(sequence) + 1):
#                     # Extract a byte from the buffer
#                     byte = buffer[i]
#                     # Check if the byte matches the sequence
#                     if byte == sequence_bytes[0]:
#                         # Verify the exact bit pattern
#                         # Convert byte to 8-bit binary string
#                         byte_bits = format(byte, '08b')
#                         # Check if the sequence matches the last 8 bits
#                         if byte_bits == sequence:
#                             return preamble_found == True
                
#                 # Keep only the overlapping part for next iteration
#                 # This ensures we don't miss sequences that span across chunks
#                 overlap = len(sequence) - 1
#                 if len(buffer) > overlap:
#                     buffer = buffer[-overlap:]
            
#             return False
            
#     except FileNotFoundError:
#         print(f"Error: File '{filename}' not found.")
#         return False
#     except Exception as e:
#         print(f"Error reading file: {e}")
#         return False


# Defining main function
def main(filename = file):
    find_sequence_in_binary_file(filename,preamble)

    # search for preamble 10011010
    if preamble_found:
        print("Sequence",preamble,"found in the file!")
    else:
        print("Sequence",preamble,"found in the file.")

    return 0

# Using the special variable 
# __name__
if __name__=="__main__":
    main()




# if preamble found - save next 112 bits
# convert to hex
# save as a message

# import pyModeS

# global msg_hex

# def extract_adsb_message(bits, start_index, preamble_found,msg):
#     """
#     Extract the 112 bits immediately following a detected preamble.

#     Parameters:
#         bits (str): Binary string containing 0s and 1s.
#         start_index (int): Index immediately after the preamble.
#         preamble_found (bool): True if a valid preamble was detected.

#     Returns:
#         str or None: 112-bit ADS-B message, or None if unavailable.
#     """
#     if preamble_found and start_index + 112 <= len(bits):
#         message = bits[start_index:start_index + 112]
#         msg_hex = format(int(message, 2), "028X")
#         return msg_hex

#     return None

# result = pyModeS.decode(msg_hex)
# print(result)
# # {
# #     'df': 17,
# #     'icao': '406B90',
# #     'crc_valid': True,
# #     'typecode': 4,
# #     'bds': '0,8',
# #     'callsign': 'EZY85MH',
# #     'category': 0,
# #     'wake_vortex': 'No category information',
# # }






















# Python code to convert binary number
# into hexadecimal number

# function to convert
# binary to hexadecimal

# def binToHexa(n):
#     bnum = int(n)
#     temp = 0
#     mul = 1
    
#     # counter to check group of 4
#     count = 1
    
#     # char array to store hexadecimal number
#     hexaDeciNum = ['0'] * 100
    
#     # counter for hexadecimal number array
#     i = 0
#     while bnum != 0:
#         rem = bnum % 10
#         temp = temp + (rem*mul)
        
#         # check if group of 4 completed
#         if count % 4 == 0:
        
#             # check if temp < 10
#             if temp < 10:
#                 hexaDeciNum[i] = chr(temp+48)
#             else:
#                 hexaDeciNum[i] = chr(temp+55)
#             mul = 1
#             temp = 0
#             count = 1
#             i = i+1
            
#         # group of 4 is not completed
#         else:
#             mul = mul*2
#             count = count+1
#         bnum = int(bnum/10)
        
#     # check if at end the group of 4 is not
#     # completed
#     if count != 1:
#         hexaDeciNum[i] = chr(temp+48)
        
#     # check at end the group of 4 is completed
#     if count == 1:
#         i = i-1
        
#     # printing hexadecimal number
#     # array in reverse order
#     print("\n Hexadecimal equivalent of {}:  ".format(n), end="")
#     while i >= 0:
#         print(end=hexaDeciNum[i])
#         i = i-1


# Final Code - python not GNU Radio

In [ ]:
"""
Embedded Python Blocks:

Each time this file is saved, GRC will instantiate the first class it finds
to get ports and parameters of your block. The arguments to __init__  will
be the parameters. All of them are required to have default values!
"""

import numpy as np

preamble_found = False
file_path = '/home/bri/workspace/mode_s_decode/controlpathsdotcom/raw_adsb_files/adsb_raw_4000000Hz_1783773546.real_float'
preamble = "1000001010000000"

def find_sequence_in_binary_file(filename, sequence):
    """Check a binary file for the presence of a specific bit sequence.
    
    Args:
        filename (str): Path to the binary file to check
        sequence (str): The bit sequence to search for (e.g., "1010001010000000")
    
    Returns:
        list: List of byte offsets where the sequence starts, or empty list if not found
    """
    print(sequence)
    if not sequence or not all(c in '01' for c in sequence):
        raise ValueError("Sequence must be a string of 0s and 1s (e.g., '10100010')")

    # Convert the bit sequence to bytes (pad to full bytes)
    num_bytes = (len(sequence) + 7) // 8
    # Pad with leading zeros if needed
    padded_sequence = sequence.zfill(num_bytes * 8)
    sequence_int = int(padded_sequence, 2)
    sequence_bytes = sequence_int.to_bytes(num_bytes, byteorder='big')

    # Also keep the original bit string for verification
    sequence_bits = sequence

    found_positions = []

    try:
        with open(filename, 'rb') as file:
            chunk_size = 65536  # 64 KB chunks
            buffer = b''
            buffer_bits = ""  # Keep track of bits for overlap

            while True:
                chunk = file.read(chunk_size)
                if not chunk:
                    break

                # Convert chunk to binary string (e.g., b'\x01\x02' → "0000000100000010")
                chunk_bits = ''.join(format(byte, '08b') for byte in chunk)

                # Combine with previous buffer's bits (for overlap)
                combined_bits = buffer_bits + chunk_bits
                # Search for the sequence in the combined bits
                start = 0
                while True:
                    pos = combined_bits.find(sequence_bits, start)
                    if pos == -1:
                        break
                    # Convert bit position to byte position
                    byte_pos = (len(buffer) * 8 + pos) // 8
                    found_positions.append(byte_pos)
                    start = pos + 1  # Continue searching for overlapping matches

                # Update buffer for next iteration (keep enough for overlap)
                buffer_bits = combined_bits[-(len(sequence_bits) - 1):]
                buffer = chunk  # Keep full chunk for byte position calculation
        

        return found_positions

    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []

def main():
    found_pos = find_sequence_in_binary_file(file_path,preamble)
    print(found_pos)

if __name__=="__main__":
    main()

In [ ]:
import numpy as np

def find_sequence_in_binary_file(filename, sequence):
    """Check if a binary file contains a specific bit sequence.
    
    Args:
        filename (str): Path to the binary file
        sequence (str): Bit sequence (e.g., "010000000000010001000000000000")
    
    Returns:
        bool: True if found, False otherwise
    """
    if not sequence or not all(c in '01' for c in sequence):
        raise ValueError("Sequence must be a string of 0s and 1s")

    # Read entire file as bytes
    try:
        with open(filename, 'rb') as f:
            data = f.read()
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return False
    except Exception as e:
        print(f"Error reading file: {e}")
        return False

    # Convert to bit string (e.g., b'\x01' → "00000001")
    bit_string = ''.join(format(byte, '08b') for byte in data)

    # Search for the sequence
    return sequence in bit_string


# Example usage
if __name__ == "__main__":
    file = '/home/bri/workspace/mode_s_decode/controlpathsdotcom/raw_adsb_files/adsb_raw_4000000Hz_1783773546.real_float'
    sequence = "1000001010000000"  # Your 28-bit sequence
    
    found = find_sequence_in_binary_file(file, sequence)
    print(f"Sequence found: {found}")

In [12]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

detections = []

# Keep enough history so packets spanning buffers aren't missed
history = np.zeros(512, dtype=np.float32)

last_detection = -100000


####################################################################
# Detect one ADS-B preamble (32 samples @ 4 MHz)
####################################################################
# def is_preamble(w):

#     if len(w) != 32:
#         return False

#     pulse = np.mean(np.concatenate((
#         w[0:2],
#         w[4:6],
#         w[14:16],
#         w[18:20]
#     )))

#     gap = np.mean(np.concatenate((
#         w[2:4],
#         w[6:14],
#         w[16:18],
#         w[20:32]
#     )))

#     if pulse < 0.02:
#         return False

#     if pulse < 3 * gap:
#         return False

#     return True
def is_preamble(w):

    if len(w) != 32:
        return False


    pulse_sections = [
        w[0:2],
        w[4:6],
        w[14:16],
        w[18:20]
    ]

    gap_sections = [
        w[2:4],
        w[6:14],
        w[16:18],
        w[20:32]
    ]


    pulse = np.mean(np.concatenate(pulse_sections))
    gap   = np.mean(np.concatenate(gap_sections))


    # absolute signal level
    if pulse < 0.25:
        return False


    # pulses must be much stronger than gaps
    if pulse < 4 * gap:
        return False


    # every pulse location should actually contain energy
    for p in pulse_sections:
        if np.mean(p) < 0.35:
            return False


    # gaps should actually be quiet
    for g in gap_sections:
        if np.mean(g) > 0.20:
            return False


    return True


####################################################################
# Fast vectorized preamble search
####################################################################


def find_preambles(samples):

    windows = sliding_window_view(samples, 32)

    pulse = (
        windows[:,0:2].mean(axis=1) +
        windows[:,4:6].mean(axis=1) +
        windows[:,14:16].mean(axis=1) +
        windows[:,18:20].mean(axis=1)
    ) / 4

    gap = (
        windows[:,2:4].mean(axis=1) +
        windows[:,6:14].mean(axis=1) +
        windows[:,16:18].mean(axis=1) +
        windows[:,20:32].mean(axis=1)
    ) / 4


    hits = np.where(
        (pulse > 0.25) &
        (pulse > 4 * gap)
    )[0]


    return hits.tolist()


####################################################################
# Process one buffer
####################################################################
def main(hist, samps, last_detect):

    global detections

    # Convert uint8 IQ magnitude to float 0-1
    samps = samps.astype(np.float32)

    if samps.max() > 1:
        samps /= 255.0

    # prepend history
    samples = np.concatenate((hist, samps))


    detections = find_preambles(samples)


    for d in detections:

        absolute_position = d - len(hist)

        if absolute_position - last_detect > 64:

            print("Detection:", absolute_position)
            print(np.round(samples[d:d+40],3))
            print()

            last_detect = absolute_position


    # save history for next buffer
    hist = samples[-512:]


    return hist, last_detect



if __name__ == "__main__":

    file = '/home/bri/workspace/mode_s_decode/controlpathsdotcom/raw_adsb_files/adsb_raw_4000000Hz_1783781711_mag_char'

    raw = np.fromfile(file, dtype=np.uint8)


    # process in chunks like an SDR stream
    chunk_size = 65536

    hist = history
    last = last_detection

    for start in range(0,len(raw),chunk_size):

        chunk = raw[start:start+chunk_size]

        hist,last = main(hist,chunk,last)

Detection: 32005
[0.984 0.867 0.227 0.024 0.769 0.976 0.227 0.106 0.024 0.384 0.227 0.714
 0.067 0.361 0.224 0.329 0.004 0.039 0.227 0.969 0.157 0.898 0.224 0.424
 0.027 0.02  0.227 0.18  0.408 0.231 0.227 0.035 0.114 0.122 0.227 0.286
 0.345 0.529 0.227 0.071]



In [13]:
#!/usr/bin/env python3
"""
adsb_preamble_finder.py

Fast ADS-B (Mode S Extended Squitter, 1090 MHz) preamble detector for
magnitude data captured from an ADALM-PLUTO SDR.

ADS-B preamble structure
-------------------------
The preamble is 8 microseconds long and contains four short pulses at:
    0.0 us, 1.0 us, 3.5 us, 4.5 us
All other time slots in the 8 us window should be quiet (low amplitude).
This script scans the magnitude waveform for that pulse pattern using a
vectorized numpy comparison, so it stays fast even on long captures.

Assumptions
-----------
* Input file is raw magnitude samples (not complex IQ) as a flat binary
  array of a fixed dtype (default: float32).
  - If you actually captured raw IQ from the Pluto, convert it to
    magnitude first, e.g.:
        iq = np.fromfile(path, dtype=np.complex64)
        mag = np.abs(iq).astype(np.float32)
        mag.tofile("mag.bin")
* Sample rate should be a Pluto capture rate that is a "nice" multiple of
  2 MHz for clean sample alignment (2 MHz, 4 MHz, 8 MHz, etc. all work
  well). Pass --rate to match your capture.

Usage
-----
    python3 adsb_preamble_finder.py capture.bin --rate 2000000

    python3 adsb_preamble_finder.py capture.bin --rate 2000000 \\
        --dtype float32 --threshold 2.0 --max-hits 20
"""

import argparse
import numpy as np


def find_preambles(mag, rate=4_000_000, threshold_ratio=2.0, min_amplitude=0.0):
    """
    Vectorized search for ADS-B preambles in a magnitude waveform.

    Parameters
    ----------
    mag : 1D numpy array of magnitude samples
    rate : sample rate in Hz used for the capture
    threshold_ratio : how many times stronger the 4 pulse samples must be
        vs. the quiet samples in the same window (higher = stricter)
    min_amplitude : absolute floor the pulses must exceed (helps ignore
        noise floor triggering false positives on quiet recordings)

    Returns
    -------
    numpy array of sample indices where a preamble was found (index marks
    the start of the preamble, i.e. the first pulse at t=0us).
    """
    samples_per_us = rate / 1_000_000.0
    if not float(samples_per_us).is_integer():
        print(f"[warn] rate {rate} Hz is not an integer number of samples "
              f"per microsecond ({samples_per_us}); pulse alignment may be "
              f"imprecise. Prefer a rate that's a whole multiple of 1 MHz.")

    pulse_us = np.array([0.0, 1.0, 3.5, 4.5])
    pulse_idx = np.round(pulse_us * samples_per_us).astype(int)
    preamble_len = int(round(8 * samples_per_us))

    n_windows = len(mag) - preamble_len
    if n_windows <= 0:
        return np.array([], dtype=int)

    # memory-efficient overlapping windows (view, not a copy)
    windows = np.lib.stride_tricks.sliding_window_view(mag, preamble_len)[:n_windows]

    # pulse (should-be-high) samples
    peaks = windows[:, pulse_idx]
    peak_min = peaks.min(axis=1)  # weakest of the 4 pulses

    # everything else in the window (should-be-low), excluding a 1-sample
    # guard band around each pulse to avoid penalizing pulse rise/fall edges
    low_mask = np.ones(preamble_len, dtype=bool)
    for i in pulse_idx:
        low_mask[max(0, i - 1): i + 2] = False
    lows = windows[:, low_mask]
    low_max = lows.max(axis=1)  # strongest "quiet" sample

    candidates = np.where(
        (peak_min > threshold_ratio * low_max) & (peak_min > min_amplitude)
    )[0]

    return candidates


def dedupe_hits(hits, min_spacing):
    """Collapse clusters of adjacent hits (from a sliding window) down to
    the strongest single detection per cluster, keeping just the first hit
    of each run that's close together."""
    if len(hits) == 0:
        return hits
    keep = [hits[0]]
    for h in hits[1:]:
        if h - keep[-1] > min_spacing:
            keep.append(h)
    return np.array(keep)


def main():
    ap = argparse.ArgumentParser(description="Find ADS-B preambles in a Pluto SDR magnitude capture.")
    ap.add_argument("infile", help="path to binary magnitude file")
    ap.add_argument("--rate", type=float, default=2_000_000,
                     help="sample rate in Hz (default: 2000000)")
    ap.add_argument("--dtype", default="float32",
                     help="numpy dtype of the samples in the file (default: float32). "
                          "Common alternatives: uint8, int16, uint16, float64")
    ap.add_argument("--threshold", type=float, default=2.0,
                     help="pulse-to-quiet amplitude ratio required (default: 2.0)")
    ap.add_argument("--min-amplitude", type=float, default=0.0,
                     help="absolute amplitude floor for pulses (default: 0, i.e. off)")
    ap.add_argument("--max-hits", type=int, default=20,
                     help="max number of hits to print (default: 20)")
    args = ap.parse_args()

    mag = np.fromfile(args.infile, dtype=np.dtype(args.dtype)).astype(np.float32)
    print(f"Loaded {len(mag):,} samples ({len(mag) / args.rate * 1000:.2f} ms) "
          f"from {args.infile}")

    hits = find_preambles(
        mag,
        rate=args.rate,
        threshold_ratio=args.threshold,
        min_amplitude=args.min_amplitude,
    )

    # a real preamble will trigger the sliding window a few samples in a
    # row; collapse those into single detections spaced at least one
    # preamble-length apart
    preamble_len = int(round(8 * args.rate / 1_000_000))
    hits = dedupe_hits(hits, min_spacing=preamble_len)

    print(f"Found {len(hits)} candidate preamble(s).")
    for h in hits[: args.max_hits]:
        t_ms = h / args.rate * 1000
        print(f"  sample {h:>10}  |  t = {t_ms:.4f} ms")

    if len(hits) > args.max_hits:
        print(f"  ... and {len(hits) - args.max_hits} more")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--rate RATE] [--dtype DTYPE]
                             [--threshold THRESHOLD]
                             [--min-amplitude MIN_AMPLITUDE]
                             [--max-hits MAX_HITS]
                             infile
ipykernel_launcher.py: error: the following arguments are required: infile


SystemExit: 2

/home/bri/miniconda3/envs/modes/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
